
# Lab 6-2 Prepare and Analyze Loan data from Lending Club

This lab guides you through exploring, preparing, and analyzing a real-world loan dataset from Lending Club. You will learn how to load and sample large datasets, create tables, and perform basic data exploration and preprocessing. By working with this data, you'll gain insights into general steps and best practices of data engineering on Databricks platforms.

This data set represents ~1.5 million loans made through the Lending Club platform, which is a platform that allows individuals to lend to other individuals. 

The data file needed for this lab

- `/databricks-datasets/samples/lending_club/` (one of the DBFS sample datasets)

Files/Tables created by this lab

- database: `lending_club`
  - table: `loans`
  - table: `loans_silver`
- volume: `msbabigdata.spark.lab6_2`
  - folder: `loan_status_by_state`

## 1.  Explore the data set

The data is part of the Databricks' collection of sample data. It is available through the path below.

In [ ]:
path = "/databricks-datasets/samples/lending_club"

### 1.1 View the folder content.

> using `dbutils.fs.ls()`

- What's in the folder? 
- What format are the data files?

In [ ]:
display(dbutils.fs.ls(path))

path | name | size | modificationTime
dbfs:/databricks-datasets/samples/lending_club/parquet/ | parquet/ | 0 | 1771313174593
dbfs:/databricks-datasets/samples/lending_club/readme.md | readme.md | 785 | 1596569741000

In [ ]:
display(dbutils.fs.ls(path+"/parquet"))

path | name | size | modificationTime
dbfs:/databricks-datasets/samples/lending_club/parquet/_SUCCESS | _SUCCESS | 0 | 1596569736000
dbfs:/databricks-datasets/samples/lending_club/parquet/_committed_4376965328215018897 | _committed_4376965328215018897 | 514 | 1596569737000
dbfs:/databricks-datasets/samples/lending_club/parquet/_started_4376965328215018897 | _started_4376965328215018897 | 0 | 1596569737000
dbfs:/databricks-datasets/samples/lending_club/parquet/part-00000-tid-4376965328215018897-67dfb805-a14c-42f6-99f0-1a6608ac4723-443-c000.snappy.parquet | part-00000-tid-4376965328215018897-67dfb805-a14c-42f6-99f0-1a6608ac4723-443-c000.snappy.parquet | 72106652 | 1596569737000
dbfs:/databricks-datasets/samples/lending_club/parquet/part-00001-tid-4376965328215018897-67dfb805-a14c-42f6-99f0-1a6608ac4723-444-c000.snappy.parquet | part-00001-tid-4376965328215018897-67dfb805-a14c-42f6-99f0-1a6608ac4723-444-c000.snappy.parquet | 57676765 | 1596569737000
dbfs:/databricks-datasets/samples/lendi

### 1.2  View the README

A readme file for this dataset has been placed in this folder.

- View the content of this file using `dbutils.fs.head()`

In [ ]:
print(dbutils.fs.head(path+'/readme.md'))

Lending Club Parquet Dataset
This data set was generated by using Apache Spark to take the Lending Club dataset from https://www.kaggle.com/wendykan/lending-club-loan-data/home and saving it in Parquet format.


Data Set Information
The original Lending Club dataset is at: https://www.kaggle.com/wendykan/lending-club-loan-data/home

	
License and/or Citation
Copyright (2019) Databricks, Inc.  This dataset is licensed under a Creative Commons Attribution 4.0 International License<https://creativecommons.org/licenses/by/4.0/>.



## 2. Load data

### 2.1 Use spark to load the data into a dataframe `loans`
- Then, verify that the loading is succeseful by displaying the first 20 and count the number of records.

At this point, the dataframe is not yet registered as a table, so we cannot yet use SQL to query it. 

- we can use `[df].limit(n).display()` to show the first n rows.

In [ ]:
loans = spark.read.parquet(path+"/parquet")

loans.limit(20).display()


id | member_id | loan_amnt | funded_amnt | funded_amnt_inv | term | int_rate | installment | grade | sub_grade | emp_title | emp_length | home_ownership | annual_inc | verification_status | loan_status | pymnt_plan | url | desc | purpose | title | zip_code | addr_state | dti | delinq_2yrs | earliest_cr_line | inq_last_6mths | mths_since_last_delinq | mths_since_last_record | open_acc | pub_rec | revol_bal | revol_util | total_acc | initial_list_status | out_prncp | out_prncp_inv | total_pymnt | total_pymnt_inv | total_rec_prncp | total_rec_int | total_rec_late_fee | recoveries | collection_recovery_fee | last_pymnt_d | last_pymnt_amnt | next_pymnt_d | last_credit_pull_d | collections_12_mths_ex_med | mths_since_last_major_derog | policy_code | application_type | annual_inc_joint | dti_joint | verification_status_joint | acc_now_delinq | tot_coll_amt | tot_cur_bal | open_acc_6m | open_il_6m | open_il_12m | open_il_24m | mths_since_rcnt_il | total_bal_il | il_util | open_rv_12m | open_rv

### 2.2 Save a small sample of data as a Table 

Exploring a large dataset can be slow and computationally expensive. Oftentimes, you would like to start small, establishing your data engineering pipeline first on a sample dataset. Then you can rerun the pipeline on the full dataset.

- please randomly sample a 1% of the dataset with a random seed of 1234 (so that it is **idempotent** -- you obtain the same sample each time)
- use `[DF].sample(pct, seed=)`


In [ ]:
loan_sample = loans.sample(0.01, seed = 1234)

Then create a database `lending_club`.

Using `[df].write.saveAsTable` to save the sample data to the table `loans` in the `lending_club` database.

- to ensure the workflow is idempotent (resuling in the same if you rerun the workflow), please use the **overwrite** mode.

In [ ]:
%sql
create database if not exists lending_club;

In [ ]:
loan_sample.write.mode("overwrite").saveAsTable('lending_club.loans')


## 3. Explore and preprocess  the data

### 3.1 Display the first 100 rows of the table

In [ ]:
%sql
select * from lending_club.loans limit 100;

id | member_id | loan_amnt | funded_amnt | funded_amnt_inv | term | int_rate | installment | grade | sub_grade | emp_title | emp_length | home_ownership | annual_inc | verification_status | loan_status | pymnt_plan | url | desc | purpose | title | zip_code | addr_state | dti | delinq_2yrs | earliest_cr_line | inq_last_6mths | mths_since_last_delinq | mths_since_last_record | open_acc | pub_rec | revol_bal | revol_util | total_acc | initial_list_status | out_prncp | out_prncp_inv | total_pymnt | total_pymnt_inv | total_rec_prncp | total_rec_int | total_rec_late_fee | recoveries | collection_recovery_fee | last_pymnt_d | last_pymnt_amnt | next_pymnt_d | last_credit_pull_d | collections_12_mths_ex_med | mths_since_last_major_derog | policy_code | application_type | annual_inc_joint | dti_joint | verification_status_joint | acc_now_delinq | tot_coll_amt | tot_cur_bal | open_acc_6m | open_il_6m | open_il_12m | open_il_24m | mths_since_rcnt_il | total_bal_il | il_util | open_rv_12m | open_rv

In [ ]:
%sql
select count(*) from lending_club.loans;

count(*)
14703

### 3.2 create a new temp view `loans1` that keeps only the following fields

> note: temp views do not belong to any database.

`loan_amnt,funded_amnt,term,int_rate,grade,annual_inc,loan_status,purpose,addr_state,issue_d,total_pymnt`



In [ ]:
%sql
create or replace temp view loans1 as 
select loan_amnt,funded_amnt,term,int_rate,grade,annual_inc,loan_status,purpose,addr_state,issue_d,total_pymnt
from lending_club.loans;




| field | type | description | 
|-- | --|--|
|loan_amnt|Float|The listed amount of the loan applied for by the borrower. If at some point in time, the credit department reduces the loan amount, then it will be reflected in this value.|
|funded_amnt|Integer|The total amount committed to that loan at that point in time.|
|term|String|The number of payments on the loan. Values are in months and can be either 36 or 60.|
|int_rate|String|Interest Rate on the loan|
|grade|String|LC assigned loan grade|
|annual_inc|Float|The self-reported annual income provided by the borrower during registration.|
|loan_status|String|Current status of the loan|
|purpose|String|A category provided by the borrower for the loan request. |
|addr_state|String|The state provided by the borrower in the loan application|
|issue_d|String|The month which the loan was funded|
|total_pymnt|String|Payments received to date for total amount funded|


Then verify the content of `loans1` by showing the first 20 rows.

In [ ]:
%sql
select * from loans1 limit 20;

loan_amnt | funded_amnt | term | int_rate | grade | annual_inc | loan_status | purpose | addr_state | issue_d | total_pymnt
3900.0 | 3900 |  36 months |  19.52% | E | 39500.0 | Fully Paid | vacation | TN | May-2014 | 5183.4416665287
20000.0 | 20000 |  60 months |  13.98% | C | 217000.0 | Fully Paid | debt_consolidation | NC | May-2014 | 22596.95
6000.0 | 6000 |  36 months |   9.17% | B | 45000.0 | Fully Paid | credit_card | NY | May-2014 | 6873.7975023057
7450.0 | 7450 |  36 months |  13.35% | C | 80000.0 | Fully Paid | credit_card | MD | May-2014 | 9098.7207565428
8000.0 | 8000 |  36 months |  10.99% | B | 80000.0 | Fully Paid | other | NY | May-2014 | 9199.94
24000.0 | 24000 |  36 months |   6.03% | A | 95000.0 | Fully Paid | credit_card | CA | May-2014 | 25873.39
1800.0 | 1800 |  36 months |  16.29% | D | 8652.0 | Fully Paid | other | OH | May-2014 | 2287.3707277484
14500.0 | 14500 |  36 months |   7.69% | A | 57500.0 | Fully Paid | credit_card | TN | May-2014 | 16255.6001815326
862

next, we are going to build a series of temp views, each one is built on the previous, each representing a step in the data engineering pipeline.

- each time, we use the previous view, add a new column (while preserving the original column), and save as a new temp view.

### 3.3 convert `issue_d` to a date format


- Add  a new date field  `issue_date` based on `issue_d`
- Using PySpark's function `to_date(colname,"MMM-yyyy")` to convert it.  The function's [documentation](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_date.html)

Save the results as the new temp view `loans2`



In [ ]:
%sql
create or replace temp view loans2 as
select * , to_date(issue_d,'MMM-yyyy') as issue_date
from loans1;
    
 

In [ ]:
%sql 
select * from loans2 limit 20;

loan_amnt | funded_amnt | term | int_rate | grade | annual_inc | loan_status | purpose | addr_state | issue_d | total_pymnt | issue_date
3900.0 | 3900 |  36 months |  19.52% | E | 39500.0 | Fully Paid | vacation | TN | May-2014 | 5183.4416665287 | 2014-05-01
20000.0 | 20000 |  60 months |  13.98% | C | 217000.0 | Fully Paid | debt_consolidation | NC | May-2014 | 22596.95 | 2014-05-01
6000.0 | 6000 |  36 months |   9.17% | B | 45000.0 | Fully Paid | credit_card | NY | May-2014 | 6873.7975023057 | 2014-05-01
7450.0 | 7450 |  36 months |  13.35% | C | 80000.0 | Fully Paid | credit_card | MD | May-2014 | 9098.7207565428 | 2014-05-01
8000.0 | 8000 |  36 months |  10.99% | B | 80000.0 | Fully Paid | other | NY | May-2014 | 9199.94 | 2014-05-01
24000.0 | 24000 |  36 months |   6.03% | A | 95000.0 | Fully Paid | credit_card | CA | May-2014 | 25873.39 | 2014-05-01
1800.0 | 1800 |  36 months |  16.29% | D | 8652.0 | Fully Paid | other | OH | May-2014 | 2287.3707277484 | 2014-05-01
14500.0 | 1450


### 3.4. Convert `term` to an integer

- create a new `term_int` field based on the `term` field which is a string; The new field should be an integer. 
- The easiest is to use Spark SQL's `translate(source_value, matching, replace)` function. The [function documentation](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.translate.html)


- Save the result to Temp View `loan3`




In [ ]:
%sql
create or replace temp view loans3 as
select *, cast(translate(term, "months", "") as int) as term_int from loans2;


In [ ]:
%sql
select * from loans3 limit 20;

loan_amnt | funded_amnt | term | int_rate | grade | annual_inc | loan_status | purpose | addr_state | issue_d | total_pymnt | issue_date | term_int
3900.0 | 3900 |  36 months |  19.52% | E | 39500.0 | Fully Paid | vacation | TN | May-2014 | 5183.4416665287 | 2014-05-01 | 36
20000.0 | 20000 |  60 months |  13.98% | C | 217000.0 | Fully Paid | debt_consolidation | NC | May-2014 | 22596.95 | 2014-05-01 | 60
6000.0 | 6000 |  36 months |   9.17% | B | 45000.0 | Fully Paid | credit_card | NY | May-2014 | 6873.7975023057 | 2014-05-01 | 36
7450.0 | 7450 |  36 months |  13.35% | C | 80000.0 | Fully Paid | credit_card | MD | May-2014 | 9098.7207565428 | 2014-05-01 | 36
8000.0 | 8000 |  36 months |  10.99% | B | 80000.0 | Fully Paid | other | NY | May-2014 | 9199.94 | 2014-05-01 | 36
24000.0 | 24000 |  36 months |   6.03% | A | 95000.0 | Fully Paid | credit_card | CA | May-2014 | 25873.39 | 2014-05-01 | 36
1800.0 | 1800 |  36 months |  16.29% | D | 8652.0 | Fully Paid | other | OH | May-2014 | 22


### 3.5 Convert `int_rate` to a float number `int_rate_float`

- again, you may use `translate` to remove the `%` part. 
- use `cast(value as FLOAT)` to convert it to a new float type

- Save the result as a new Temp View `loans4`



In [ ]:
%sql
create or replace temp view loans4 as
select*, cast(translate(int_rate, "%", "") as float) as int_rate_float
from loans3;
    


In [ ]:
%sql
select* from loans4 limit 20;

loan_amnt | funded_amnt | term | int_rate | grade | annual_inc | loan_status | purpose | addr_state | issue_d | total_pymnt | issue_date | term_int | int_rate_float
3900.0 | 3900 |  36 months |  19.52% | E | 39500.0 | Fully Paid | vacation | TN | May-2014 | 5183.4416665287 | 2014-05-01 | 36 | 19.52
20000.0 | 20000 |  60 months |  13.98% | C | 217000.0 | Fully Paid | debt_consolidation | NC | May-2014 | 22596.95 | 2014-05-01 | 60 | 13.98
6000.0 | 6000 |  36 months |   9.17% | B | 45000.0 | Fully Paid | credit_card | NY | May-2014 | 6873.7975023057 | 2014-05-01 | 36 | 9.17
7450.0 | 7450 |  36 months |  13.35% | C | 80000.0 | Fully Paid | credit_card | MD | May-2014 | 9098.7207565428 | 2014-05-01 | 36 | 13.35
8000.0 | 8000 |  36 months |  10.99% | B | 80000.0 | Fully Paid | other | NY | May-2014 | 9199.94 | 2014-05-01 | 36 | 10.99
24000.0 | 24000 |  36 months |   6.03% | A | 95000.0 | Fully Paid | credit_card | CA | May-2014 | 25873.39 | 2014-05-01 | 36 | 6.03
1800.0 | 1800 |  36 months |

### 3.6 Inspect the final view `loans4` 

- First, view the first 20 rows from loans 4
- Then, calculate summary statistics of the columns using DataFrame API  `.describe()`

In [ ]:
spark.table("loans4").describe().display()

summary | loan_amnt | funded_amnt | term | int_rate | grade | annual_inc | loan_status | purpose | addr_state | issue_d | total_pymnt | term_int | int_rate_float
count | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703 | 14703
mean | 14775.113922328777 | 14775.113922328777 | None | None | None | 77773.84430333892 | None | None | None | None | 10501.309324749027 | 42.70230565190777 | 13.143801865736286
stddev | 8730.040980661259 | 8730.040980661259 | None | None | None | 86134.41253932824 | None | None | None | None | 8993.92841273093 | 10.767651564642511 | 4.680196576762972
min | 1000.0 | 1000 |  36 months |   5.32% | A | 0.0 | Charged Off | car | AK | Apr-2012 | 0.0 | 36 | 5.32
max | 40000.0 | 40000 |  60 months |  30.99% | G | 8253000.0 | Late (31-120 days) | wedding | WY | Sep-2016 | 9995.74 | 60 | 30.99

### 3.7 Remove the unwanted columns and save it as Temp View `loans5`

- We will keep these fields `loan_amnt,funded_amnt,term_int,int_rate_float,grade,annual_inc,loan_status,purpose,addr_state,issue_date,total_pymnt`


In [ ]:
%sql
create or replace temp view loans5 as 
select loan_amnt,funded_amnt,term_int,int_rate_float,grade,annual_inc,loan_status,purpose,addr_state,issue_date,total_pymnt
from loans4;

In [ ]:
%sql
select* from loans5 limit 20;

loan_amnt | funded_amnt | term_int | int_rate_float | grade | annual_inc | loan_status | purpose | addr_state | issue_date | total_pymnt
3900.0 | 3900 | 36 | 19.52 | E | 39500.0 | Fully Paid | vacation | TN | 2014-05-01 | 5183.4416665287
20000.0 | 20000 | 60 | 13.98 | C | 217000.0 | Fully Paid | debt_consolidation | NC | 2014-05-01 | 22596.95
6000.0 | 6000 | 36 | 9.17 | B | 45000.0 | Fully Paid | credit_card | NY | 2014-05-01 | 6873.7975023057
7450.0 | 7450 | 36 | 13.35 | C | 80000.0 | Fully Paid | credit_card | MD | 2014-05-01 | 9098.7207565428
8000.0 | 8000 | 36 | 10.99 | B | 80000.0 | Fully Paid | other | NY | 2014-05-01 | 9199.94
24000.0 | 24000 | 36 | 6.03 | A | 95000.0 | Fully Paid | credit_card | CA | 2014-05-01 | 25873.39
1800.0 | 1800 | 36 | 16.29 | D | 8652.0 | Fully Paid | other | OH | 2014-05-01 | 2287.3707277484
14500.0 | 14500 | 36 | 7.69 | A | 57500.0 | Fully Paid | credit_card | TN | 2014-05-01 | 16255.6001815326
8625.0 | 8625 | 36 | 14.99 | C | 55969.0 | Fully Paid | d

### 3.8 Save loan5 as `loans_silver` table in `lending_club`.

- By saving temp view as a table, we persist the preprocessed data on disk so that we don't have to recompute it in the future each time. 
- `bronze`, `silver`, `gold` is Databrick's language for describing the phase of the dataset, where bronze is the raw (uncleaned) data and `gold` is the final dataset for end users.


In [ ]:
spark.table("loans5").write.mode("overwrite").saveAsTable("lending_club.loans_silver")


## 4. Query and Insights

### 4.1 Count the loan by loan status 
- Report the results by descending order of the counts.
- Save results in a Temp View `loan_status_by_state`

In [ ]:
%sql
create or replace temp view loan_status_by_state as
select loan_status, count(*) as count
from lending_club.loans_silver
group by loan_status
order by 'count' desc;

In [ ]:
%sql
select * from loan_status_by_state;
    


loan_status | count
Current | 7854
Fully Paid | 5206
In Grace Period | 117
Charged Off | 1269
Late (31-120 days) | 213
Late (16-30 days) | 44

### 4.2 Export `loan_status_by_state` to a CSV file

Sometimes, our aggregation results may need to be saved for consumption by other tools or users. 
We will save it in the format that's easy to share.
- First, create the catalog (`msbabigdata`), schema (`spark`), and volume (`lab6_2 `)
- Save it under a folder `loan_status_by_state` in Lab6_2, with header.

In [ ]:
%sql
create catalog if not exists msbabigdata;
create database if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.lab6_2;


    


In [ ]:
spark.table("loan_status_by_state").write.mode("overwrite").csv("/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state")


### 4.3 Verify the files in the output folder

In [ ]:
path = '/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state'
display(dbutils.fs.ls(path))


path | name | size | modificationTime
dbfs:/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state/_SUCCESS | _SUCCESS | 0 | 1771313410000
dbfs:/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state/_committed_3356344086163693654 | _committed_3356344086163693654 | 113 | 1771313392000
dbfs:/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state/_committed_6879448705575950808 | _committed_6879448705575950808 | 212 | 1771313409000
dbfs:/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state/_started_3356344086163693654 | _started_3356344086163693654 | 0 | 1771313391000
dbfs:/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state/_started_6879448705575950808 | _started_6879448705575950808 | 0 | 1771313409000
dbfs:/Volumes/msbabigdata/spark/lab6_2/loan_status_by_state/part-00000-tid-6879448705575950808-4e43eb34-57f9-482e-97f9-e786db15101c-244-1-c000.csv | part-00000-tid-6879448705575950808-4e43eb34-57f9-482e-97f9-e786db15101c-244-1-c000.csv | 110 | 1771313409000

In [ ]:
print(dbutils.fs.head(path+"/part-00000-tid-6879448705575950808-4e43eb34-57f9-482e-97f9-e786db15101c-244-1-c000.csv"))

In Grace Period,117
Late (16-30 days),44
Charged Off,1269
Fully Paid,5206
Late (31-120 days),213
Current,7854



## 5. Clean up

Think about what we have created, and what we need to remove, and how you remove them.

In [ ]:
%sql 
drop database if exists lending_club cascade;

In [ ]:
%sql 
drop volume if exists msbabigdata.spark.lab6_2;

This is the end of the Lab